In [77]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import ipywidgets as widgets
from IPython.display import display, clear_output
from itertools import combinations

df = pd.read_csv(
    "international-visitors-london.csv",
    encoding="cp1252"
)

In [22]:
numeric_columns = [
    "Visits (000s)",
    "Spend (£m)",
    "Nights (000s)",
    "sample"
]

In [23]:
df.head

<bound method NDFrame.head of         year        quarter        market     dur_stay mode        purpose  \
0       2002  January-March       Belgium  1-3  nights  Air        Holiday   
1       2002  January-March       Belgium  1-3  nights  Air       Business   
2       2002  January-March       Belgium  1-3  nights  Air            VFR   
3       2002  January-March       Belgium  1-3  nights  Air  Miscellaneous   
4       2002  January-March       Belgium  1-3  nights  Sea       Business   
...      ...            ...           ...          ...  ...            ...   
61457  2020P  January-March  Other Africa  4-7  nights  Air  Miscellaneous   
61458  2020P  January-March  Other Africa  8-14 nights  Air        Holiday   
61459  2020P  January-March  Other Africa  8-14 nights  Air            VFR   
61460  2020P  January-March  Other Africa  15+  nights  Air        Holiday   
61461  2020P  January-March  Other Africa  15+  nights  Air            VFR   

          area  Visits (000s)  Sp

In [24]:
df.shape

(61462, 11)

In [25]:
df.columns

Index(['year', 'quarter', 'market', 'dur_stay', 'mode', 'purpose', 'area',
       'Visits (000s)', 'Spend (£m)', 'Nights (000s)', 'sample'],
      dtype='object')

In [26]:
df.dtypes

year              object
quarter           object
market            object
dur_stay          object
mode              object
purpose           object
area              object
Visits (000s)    float64
Spend (£m)       float64
Nights (000s)    float64
sample             int64
dtype: object

DATA PREPARATION 

Unique Values

In [27]:
def display_unique_summary(df, columns):
    missing_columns = [
        column for column in columns
        if column not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Columns not found in DataFrame: {missing_columns}"
        )

    for column in columns:
        unique_values = (
            df[column]
            .dropna()
            .drop_duplicates()
            .tolist()
        )

        unique_count = len(unique_values)

        print(f"{column}:")
        print(f"Unique values: {unique_count}")

        if unique_count <= 10:
            print(", ".join(map(str, unique_values)))
        else:
            print(", ".join(map(str, unique_values[:5])))
            print("...")
            print(", ".join(map(str, unique_values[-5:])))

        print("-" * 50)

    return None


display_unique_summary(
    df,
    columns=["dur_stay", "mode", "quarter", "area", "market"]
)

dur_stay:
Unique values: 4
1-3  nights, 4-7  nights, 8-14 nights, 15+  nights
--------------------------------------------------
mode:
Unique values: 3
Air, Sea, Tunnel
--------------------------------------------------
quarter:
Unique values: 4
January-March, April-June, July-September, October-December
--------------------------------------------------
area:
Unique values: 1
 LONDON
--------------------------------------------------
market:
Unique values: 62
Belgium, Luxembourg, France, Germany, Italy
...
Chile, Indonesia, Bahrain, Oman, Qatar
--------------------------------------------------


Missing Values

In [28]:
df.isnull().sum()

year             0
quarter          0
market           0
dur_stay         0
mode             0
purpose          0
area             0
Visits (000s)    0
Spend (£m)       0
Nights (000s)    0
sample           0
dtype: int64

Unexpected Values

In [29]:
for column in numeric_columns:
    converted_values = pd.to_numeric(df[column], errors="coerce")

    strange_mask = df[column].notna() & converted_values.isna()

    print(f"\nColumn: {column}")
    print(f"Current data type: {df[column].dtype}")
    print(f"Number of strange values: {strange_mask.sum()}")

    if strange_mask.sum() > 0:
        print("Strange values:")
        print(df.loc[strange_mask, column].value_counts().head(20))


Column: Visits (000s)
Current data type: float64
Number of strange values: 0

Column: Spend (£m)
Current data type: float64
Number of strange values: 0

Column: Nights (000s)
Current data type: float64
Number of strange values: 0

Column: sample
Current data type: int64
Number of strange values: 0


In [30]:
for column in numeric_columns:
    strange_symbols = (
        df[column]
        .astype(str)
        .str.contains(r"[^0-9.,\-\s]", regex=True, na=False)
    )

    print(f"\n{column}: {strange_symbols.sum()} values with unusual symbols")

    if strange_symbols.any():
        print(df.loc[strange_symbols, column].value_counts().head(20))


Visits (000s): 0 values with unusual symbols

Spend (£m): 3 values with unusual symbols
Spend (£m)
0.000072    1
0.000057    1
0.000006    1
Name: count, dtype: int64

Nights (000s): 0 values with unusual symbols

sample: 0 values with unusual symbols


Outliers

In [31]:
for column in numeric_columns:
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df[
        (df[column] < lower_bound) |
        (df[column] > upper_bound)
    ]

    print(f"\nColumn: {column}")
    print(f"Lower bound: {lower_bound:.2f}")
    print(f"Upper bound: {upper_bound:.2f}")
    print(f"Number of outliers: {len(outliers)}")


Column: Visits (000s)
Lower bound: -4.57
Upper bound: 9.81
Number of outliers: 7054

Column: Spend (£m)
Lower bound: -3.71
Upper bound: 7.04
Number of outliers: 6274

Column: Nights (000s)
Lower bound: -34.47
Upper bound: 67.86
Number of outliers: 6757

Column: sample
Lower bound: -5.00
Upper bound: 11.00
Number of outliers: 6994


In [32]:
df.nlargest(20, "Spend (£m)")[
    [
        "year",
        "quarter",
        "market",
        "purpose",
        "Visits (000s)",
        "Spend (£m)",
        "Nights (000s)",
        "sample"
    ]
]

,year,quarter,market,purpose,Visits (000s),Spend (£m),Nights (000s),sample
28397,2010,April-June,Mexico,Holiday,7.290278,373.232590,20.387059,5
53969,2017,July-September,Saudi Arabia,Holiday,25.175373,313.081236,279.743915,26
53975,2017,July-September,Saudi Arabia,VFR,5.777371,277.357109,351.472595,4
55977,2018,April-June,USA,Holiday,142.202996,196.159302,732.787979,146
51225,2016,October-December,USA,Business,80.345259,177.243672,369.025526,86
56743,2018,July-September,USA,Holiday,182.477796,163.517932,928.712715,148
53656,2017,July-September,USA,Holiday,113.790236,162.923637,792.765753,125
24994,2009,April-June,Japan,Holiday,9.621325,155.645995,20.061919,18
55254,2018,January-March,USA,Business,107.121247,145.103330,487.949356,106
49456,2016,April-June,USA,Business,79.664955,139.922257,354.560301,129


Number Formatting

In [37]:
def format_large_number(value, position=None):
    absolute_value = abs(value)

    if absolute_value >= 1_000_000_000:
        return f"{value / 1_000_000_000:.1f}B"

    if absolute_value >= 1_000_000:
        return f"{value / 1_000_000:.1f}M"

    if absolute_value >= 1_000:
        return f"{value / 1_000:.1f}K"

    return f"{value:.0f}"


def format_currency(value, position=None):
    absolute_value = abs(value)

    if absolute_value >= 1_000_000_000:
        return f"£{value / 1_000_000_000:.1f}B"

    if absolute_value >= 1_000_000:
        return f"£{value / 1_000_000:.1f}M"

    if absolute_value >= 1_000:
        return f"£{value / 1_000:.1f}K"

    return f"£{value:.0f}"

DATA ANALISYS

Yearly Summaries

In [86]:
def create_yearly_summary(df):
    required_columns = [
        "year",
        "sample",
        "Visits (000s)",
        "Nights (000s)",
        "Spend (£m)"
    ]

    missing_columns = [
        column
        for column in required_columns
        if column not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Missing required columns: {missing_columns}"
        )

    data = df.copy()

    data["year"] = (
        data["year"]
        .astype(str)
        .str.strip()
        .str.replace("P", "", regex=False)
        .astype(int)
    )

    yearly_summary = (
        data.groupby("year", as_index=False)
        .agg(
            survey_respondents=("sample", "sum"),
            estimated_visits=("Visits (000s)", "sum"),
            estimated_nights=("Nights (000s)", "sum"),
            estimated_spending_gbp=("Spend (£m)", "sum")
        )
    )

    yearly_summary["estimated_visits"] *= 1_000
    yearly_summary["estimated_nights"] *= 1_000
    yearly_summary["estimated_spending_gbp"] *= 1_000_000

    return yearly_summary

In [87]:
def plot_yearly_metric(yearly_summary, metric):

    metric_settings = {
        "survey_respondents": {
            "title": "Survey Respondents by Year",
            "ylabel": "Survey Respondents",
            "formatter": format_large_number
        },
        "estimated_visits": {
            "title": "Estimated Visits by Year",
            "ylabel": "Estimated Visits",
            "formatter": format_large_number
        },
        "estimated_nights": {
            "title": "Estimated Nights by Year",
            "ylabel": "Estimated Nights",
            "formatter": format_large_number
        },
        "estimated_spending_gbp": {
            "title": "Estimated Visitor Spending by Year",
            "ylabel": "Estimated Spending",
            "formatter": format_currency
        }
    }

    if metric not in metric_settings:
        raise ValueError(
            f"Invalid metric. Choose one of: {list(metric_settings.keys())}"
        )

    settings = metric_settings[metric]

    fig, ax = plt.subplots(figsize=(12, 6))

    ax.bar(
        yearly_summary["year"].astype(str),
        yearly_summary[metric]
    )

    ax.set_title(settings["title"])
    ax.set_xlabel("Year")
    ax.set_ylabel(settings["ylabel"])

    ax.yaxis.set_major_formatter(
        FuncFormatter(settings["formatter"])
    )

    ax.tick_params(axis="x", rotation=45)

    plt.tight_layout()
    plt.show()

In [88]:

def format_large_number(value, position=None):
    absolute_value = abs(value)

    if absolute_value >= 1_000_000_000:
        return f"{value / 1_000_000_000:.1f}B"

    if absolute_value >= 1_000_000:
        return f"{value / 1_000_000:.1f}M"

    if absolute_value >= 1_000:
        return f"{value / 1_000:.1f}K"

    return f"{value:.0f}"


def create_market_summary(
    df,
    metric="estimated_spending_gbp",
    top_n=5,
    exclude_zero_values=False
):
    data = df.copy()

    data["market"] = data["market"].astype(str).str.strip()

    market_summary = (
        data.groupby("market", as_index=False)
        .agg(
            survey_respondents=("sample", "sum"),
            estimated_visits=("Visits (000s)", "sum"),
            estimated_nights=("Nights (000s)", "sum"),
            estimated_spending_gbp=("Spend (£m)", "sum")
        )
    )

    market_summary["estimated_visits"] *= 1_000
    market_summary["estimated_nights"] *= 1_000
    market_summary["estimated_spending_gbp"] *= 1_000_000

    ranking_data = market_summary.copy()

    if exclude_zero_values:
        ranking_data = ranking_data[ranking_data[metric] > 0]

    best_markets = (
        ranking_data
        .nlargest(top_n, metric)
        .sort_values(metric, ascending=False)
        .reset_index(drop=True)
    )

    worst_markets = (
        ranking_data
        .nsmallest(top_n, metric)
        .sort_values(metric, ascending=True)
        .reset_index(drop=True)
    )

    metric_labels = {
        "survey_respondents": "Survey Respondents",
        "estimated_visits": "Estimated Visits",
        "estimated_nights": "Estimated Nights",
        "estimated_spending_gbp": "Estimated Spending (£)"
    }

    ylabel = metric_labels[metric]

    for market_data, title_type in [
        (best_markets, "Top"),
        (worst_markets, "Bottom")
    ]:
        fig, ax = plt.subplots(figsize=(10, 6))

        ax.bar(
            market_data["market"],
            market_data[metric]
        )

        ax.set_title(f"{title_type} {top_n} Markets by {ylabel}")
        ax.set_xlabel("Market")
        ax.set_ylabel(ylabel)

        ax.yaxis.set_major_formatter(
            FuncFormatter(format_large_number)
        )

        ax.tick_params(axis="x", rotation=45)

        plt.tight_layout()
        plt.show()

    return market_summary, best_markets, worst_markets

In [89]:
def create_interactive_yearly_dashboard(df):
    yearly_summary = create_yearly_summary(df)

    metric_options = {
        "Visits": "estimated_visits",
        "Nights": "estimated_nights",
        "Respondents": "survey_respondents",
        "Spending": "estimated_spending_gbp"
    }

    metric_buttons = widgets.ToggleButtons(
        options=metric_options,
        value="estimated_visits",
        description="Metric:",
        style={"description_width": "initial"}
    )

    output = widgets.Output()

    def update_dashboard(change=None):
        with output:
            clear_output(wait=True)

            plot_yearly_metric(
                yearly_summary=yearly_summary,
                metric=metric_buttons.value
            )

    metric_buttons.observe(
        update_dashboard,
        names="value"
    )

    display(metric_buttons, output)

    update_dashboard()

    return yearly_summary
yearly_summary = create_interactive_yearly_dashboard(df)

ToggleButtons(description='Metric:', options={'Visits': 'estimated_visits', 'Nights': 'estimated_nights', 'Res…

Output()

Markets by features

In [42]:
def create_interactive_market_dashboard(
    df,
    top_n=5,
    exclude_zero_values=False
):

    metric_options = {
        "Estimated Visits": "estimated_visits",
        "Estimated Nights": "estimated_nights",
        "Survey Respondents": "survey_respondents",
        "Estimated Spending (£)": "estimated_spending_gbp"
    }

    metric_dropdown = widgets.Dropdown(
        options=metric_options,
        value="estimated_visits",
        description="Metric:",
        style={"description_width": "initial"},
        layout=widgets.Layout(width="350px")
    )

    output = widgets.Output()

    def update_dashboard(change=None):

        with output:
            clear_output(wait=True)

            market_summary, best_markets, worst_markets = (
                create_market_summary(
                    df=df,
                    metric=metric_dropdown.value,
                    top_n=top_n,
                    exclude_zero_values=exclude_zero_values
                )
            )

            print(f"Selected metric: {metric_dropdown.label}")

            # display(best_markets)
            # display(worst_markets)

    metric_dropdown.observe(
        update_dashboard,
        names="value"
    )

    display(metric_dropdown, output)

    update_dashboard()
create_interactive_market_dashboard(
    df,
    top_n=5,
    exclude_zero_values=True
)

Dropdown(description='Metric:', layout=Layout(width='350px'), options={'Estimated Visits': 'estimated_visits',…

Output()

Visits by features

In [50]:
def prepare_visitors_by_feature(df, feature):
    required_columns = [
        feature,
        "Visits (000s)"
    ]

    missing_columns = [
        column
        for column in required_columns
        if column not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Missing required columns: {missing_columns}"
        )

    data = df.copy()

    data["estimated_visits"] = (
        pd.to_numeric(data["Visits (000s)"], errors="coerce")
        * 1_000
    )

    if feature == "sample":
        data["sample"] = pd.to_numeric(
            data["sample"],
            errors="coerce"
        )

        return (
            data[["sample", "estimated_visits"]]
            .dropna()
            .reset_index(drop=True)
        )

    data[feature] = (
        data[feature]
        .astype(str)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

    feature_summary = (
        data.groupby(feature, as_index=False)
        .agg(
            estimated_visits=("estimated_visits", "sum")
        )
    )

    return feature_summary

In [51]:
def plot_visitors_by_feature(df, feature):
    feature_settings = {
        "quarter": {
            "title": "Estimated Visits by Quarter",
            "xlabel": "Quarter",
            "chart_type": "bar",
            "order": [
                "January-March",
                "April-June",
                "July-September",
                "October-December"
            ]
        },
        "dur_stay": {
            "title": "Estimated Visits by Duration of Stay",
            "xlabel": "Duration of Stay",
            "chart_type": "bar",
            "order": [
                "1-3 nights",
                "4-7 nights",
                "8-14 nights",
                "15+ nights"
            ]
        },
        "purpose": {
            "title": "Estimated Visits by Purpose",
            "xlabel": "Purpose",
            "chart_type": "bar",
            "order": None
        },
        "sample": {
            "title": "Survey Sample Size vs Estimated Visits",
            "xlabel": "Survey Respondents",
            "chart_type": "scatter",
            "order": None
        }
    }

    if feature not in feature_settings:
        raise ValueError(
            f"Invalid feature. Choose one of: "
            f"{list(feature_settings.keys())}"
        )

    settings = feature_settings[feature]

    plot_data = prepare_visitors_by_feature(
        df=df,
        feature=feature
    )

    fig, ax = plt.subplots(figsize=(11, 6))

    if settings["chart_type"] == "scatter":
        ax.scatter(
            plot_data["sample"],
            plot_data["estimated_visits"],
            alpha=0.5
        )

        ax.xaxis.set_major_formatter(
            FuncFormatter(format_large_number)
        )

    else:
        if settings["order"] is not None:
            plot_data[feature] = pd.Categorical(
                plot_data[feature],
                categories=settings["order"],
                ordered=True
            )

            plot_data = (
                plot_data
                .sort_values(feature)
                .dropna(subset=[feature])
            )
        else:
            plot_data = plot_data.sort_values(
                "estimated_visits",
                ascending=False
            )

        ax.bar(
            plot_data[feature].astype(str),
            plot_data["estimated_visits"]
        )

        ax.tick_params(
            axis="x",
            rotation=45
        )

    ax.set_title(settings["title"])
    ax.set_xlabel(settings["xlabel"])
    ax.set_ylabel("Estimated Visits")

    ax.yaxis.set_major_formatter(
        FuncFormatter(format_large_number)
    )

    plt.tight_layout()
    plt.show()

In [52]:
def create_interactive_visitors_dashboard(df):
    feature_options = {
        "Quarter": "quarter",
        "Duration of Stay": "dur_stay",
        "Purpose": "purpose",
        "Sample Size": "sample"
    }

    feature_buttons = widgets.ToggleButtons(
        options=feature_options,
        value="quarter",
        description="Compare by:",
        style={
            "description_width": "initial"
        }
    )

    output = widgets.Output()

    def update_dashboard(change=None):
        with output:
            clear_output(wait=True)

            plot_visitors_by_feature(
                df=df,
                feature=feature_buttons.value
            )

    feature_buttons.observe(
        update_dashboard,
        names="value"
    )

    display(feature_buttons, output)

    update_dashboard()
create_interactive_visitors_dashboard(df)

ToggleButtons(description='Compare by:', options={'Quarter': 'quarter', 'Duration of Stay': 'dur_stay', 'Purpo…

Output()

Corrolations

In [85]:
import pandas as pd


def display_feature_relationship(
    df,
    feature_1,
    feature_2,
    normalize=None,
    include_totals=False
):
    missing_columns = [
        column
        for column in [feature_1, feature_2]
        if column not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Columns not found in DataFrame: {missing_columns}"
        )

    normalize_options = {
        None: False,
        "row": "index",
        "column": "columns",
        "all": "all"
    }

    if normalize not in normalize_options:
        raise ValueError(
            'normalize must be None, "row", "column", or "all".'
        )

    table = pd.crosstab(
        index=df[feature_1],
        columns=df[feature_2],
        normalize=normalize_options[normalize],
        margins=include_totals,
        dropna=False
    )

    if normalize is not None:
        table = (table * 100).round(2)

    display(table)

    return table

In [82]:
def plot_relationship_heatmap(
    df,
    feature_1,
    feature_2,
    calculation="row",
    show_values=True,
    figsize=(10, 6),
):

    missing_columns = [
        column
        for column in [feature_1, feature_2]
        if column not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Columns not found in DataFrame: {missing_columns}"
        )

    valid_calculations = {"count", "row", "column", "all"}

    calculation = calculation.lower()

    if calculation not in valid_calculations:
        raise ValueError(
            f"calculation must be one of: {sorted(valid_calculations)}"
        )

    table = pd.crosstab(
        index=df[feature_1],
        columns=df[feature_2],
        dropna=False,
    )

    is_percentage = calculation != "count"

    if calculation == "row":
        table = table.div(table.sum(axis=1), axis=0) * 100

    elif calculation == "column":
        table = table.div(table.sum(axis=0), axis=1) * 100

    elif calculation == "all":
        table = table / table.to_numpy().sum() * 100

    table = table.fillna(0)

    fig, ax = plt.subplots(figsize=figsize)

    image = ax.imshow(table.to_numpy(), aspect="auto")

    ax.set_xticks(range(len(table.columns)))
    ax.set_xticklabels(
        table.columns,
        rotation=45,
        ha="right",
    )

    ax.set_yticks(range(len(table.index)))
    ax.set_yticklabels(table.index)

    ax.set_xlabel(feature_2)
    ax.set_ylabel(feature_1)
    ax.set_title(
        f"{feature_1} by {feature_2} — {calculation}"
    )

    if show_values:
        for row_index in range(table.shape[0]):
            for column_index in range(table.shape[1]):
                value = table.iloc[row_index, column_index]

                if is_percentage:
                    label = f"{value:.1f}%"
                else:
                    label = f"{value:.0f}"

                ax.text(
                    column_index,
                    row_index,
                    label,
                    ha="center",
                    va="center",
                )

    colorbar = fig.colorbar(image, ax=ax)
    colorbar.set_label(
        "Percentage" if is_percentage else "Count"
    )

    plt.tight_layout()
    plt.show()

    return table

In [83]:
def create_heatmap_picker(
    df,
    columns,
    calculations=("row", "column", "count"),
    default_calculation="row"
):
    missing_columns = [
        column for column in columns
        if column not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Columns not found in DataFrame: {missing_columns}"
        )

    if len(columns) < 2:
        raise ValueError("At least two columns must be provided.")
    column_pairs = list(combinations(columns, 2))

    pair_options = {
        f"{first_column} vs {second_column}": (
            first_column,
            second_column
        )
        for first_column, second_column in column_pairs
    }

    pair_dropdown = widgets.Dropdown(
        options=pair_options,
        description="Relationship:",
        layout=widgets.Layout(width="450px")
    )

    calculation_dropdown = widgets.Dropdown(
        options=calculations,
        value=default_calculation,
        description="Calculation:",
        layout=widgets.Layout(width="300px")
    )

    output = widgets.Output()
    result = {"table": None}

    def update_heatmap(change=None):
        first_column, second_column = pair_dropdown.value
        calculation = calculation_dropdown.value

        with output:
            clear_output(wait=True)

            print(
                f"{first_column} vs {second_column} "
                f"— calculation: {calculation}"
            )

            result["table"] = plot_relationship_heatmap(
                df,
                first_column,
                second_column,
                calculation=calculation
            )

    pair_dropdown.observe(update_heatmap, names="value")
    calculation_dropdown.observe(update_heatmap, names="value")

    display(
        widgets.VBox([
            pair_dropdown,
            calculation_dropdown,
            output
        ])
    )

    update_heatmap()

    return result

In [84]:
heatmap_result = create_heatmap_picker(
    df,
    columns=[
        "mode",
        "purpose",
        "quarter",
        "dur_stay",
        "year"
    ],
    calculations=["row", "column", "count"]
)

PREDICTION ALGORITHMS